### Item Based Collaborative Filtering (Recommender System)

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
ratings = pd.read_csv("mlsmall/ratings.csv")
movies = pd.read_csv("mlsmall/movies.csv")

TOP_K = 20
TOP_N = 10

DEMO_USER_ID = 25
DEMO_MOVIE_ID = 1

In [9]:
print(movies.shape)
print(ratings.shape)

(9742, 3)
(100836, 4)


In [19]:
movie_titles = movies.set_index("movieId")["title"]
movie_titles.unique

<bound method Series.unique of movieId
1                                  Toy Story (1995)
2                                    Jumanji (1995)
3                           Grumpier Old Men (1995)
4                          Waiting to Exhale (1995)
5                Father of the Bride Part II (1995)
                            ...                    
193581    Black Butler: Book of the Atlantic (2017)
193583                 No Game No Life: Zero (2017)
193585                                 Flint (2017)
193587          Bungo Stray Dogs: Dead Apple (2018)
193609          Andrew Dice Clay: Dice Rules (1991)
Name: title, Length: 9742, dtype: object>

In [23]:
train_data, test_data = train_test_split(ratings, test_size=0.20, random_state=19)

print("\nTraining ratings :", len(train_data))
print("Testing ratings  :", len(test_data))


Training ratings : 80668
Testing ratings  : 20168


In [26]:
train_matrix = train_data.pivot_table(index="userId", columns="movieId", values="rating")

print("\nTraining user-item matrix shape:")
print(train_matrix.shape)
print(train_matrix.isna().sum().sum())


Training user-item matrix shape:
(610, 8937)
5370902


STEP 1 - USER Based Collaborative Filtering - Comparing Columns/Items

In [27]:
item_user_matrix = train_matrix.T
item_user_filled = item_user_matrix.fillna(0).astype(np.float32)

print("\nItem-user matrix shape:")
print(item_user_matrix.shape)


Item-user matrix shape:
(8937, 610)


STEP 2 - Calculate Similarity between Items (Columns) using Cosine Similarity

In [28]:
item_similarity = cosine_similarity(item_user_filled)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_filled.index,
    columns=item_user_filled.index
)

print("\nItem-item similarity matrix created.")
print("Shape:", item_similarity_df.shape)


Item-item similarity matrix created.
Shape: (8937, 8937)


STEP 3 - Recommend Movies Similar to a Selected Movie

In [29]:
def show_similar_movies(movie_id, top_n=10):

    if movie_id not in item_similarity_df.index: return pd.DataFrame() # empty frame

    similarities = (item_similarity_df[movie_id].drop(movie_id).sort_values(ascending=False).head(top_n))

    result = pd.DataFrame({ "movieId": similarities.index, "Similarity": similarities.values})

    result["Movie"] = result["movieId"].map(movie_titles)

    return result[["movieId", "Movie", "Similarity"]]

In [30]:
print("\n" + "=" * 70)
print("SIMILAR MOVIES")
print("=" * 70)

print("Target Movie:", movie_titles.get(DEMO_MOVIE_ID, DEMO_MOVIE_ID))


SIMILAR MOVIES
Target Movie: Toy Story (1995)


In [31]:
display(show_similar_movies(DEMO_MOVIE_ID, top_n=10))

,movieId,Movie,Similarity
0,1073,Willy Wonka & the Chocolate Factory (1971),0.480831
1,780,Independence Day (a.k.a. ID4) (1996),0.465436
2,1210,Star Wars: Episode VI - Return of the Jedi (1983),0.455110
3,480,Jurassic Park (1993),0.447238
4,648,Mission: Impossible (1996),0.441350
5,356,Forrest Gump (1994),0.440126
6,34,Babe (1995),0.438336
7,608,Fargo (1996),0.434825
8,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.433407
9,260,Star Wars: Episode IV - A New Hope (1977),0.430651


STEP 4 - Item Based Rating Prediction

In [33]:
def predict_basic_item_cf(user_id, movie_id, top_k=TOP_K):

    if user_id not in train_matrix.index: return np.nan

    if movie_id not in item_similarity_df.index: return np.nan

    similarities = ( item_similarity_df[movie_id].drop(movie_id).sort_values(ascending=False).head(top_k))

    weighted_sum = 0.0
    similarity_sum = 0.0

    for neighbor_movie in similarities.index:

        rating = train_matrix.loc[user_id, neighbor_movie]

        if pd.notna(rating):

            similarity = similarities[neighbor_movie]

            if similarity > 0:
                weighted_sum += similarity * rating
                similarity_sum += abs(similarity)

    if similarity_sum == 0:
        return np.nan

    prediction = weighted_sum / similarity_sum

    return np.clip(prediction, 0.5, 5.0)

In [34]:
prediction = predict_basic_item_cf(
    DEMO_USER_ID,
    DEMO_MOVIE_ID
)

print("\n" + "=" * 70)
print("SINGLE RATING PREDICTION")
print("=" * 70)
print("User :", DEMO_USER_ID)
print("Movie:", movie_titles.get(DEMO_MOVIE_ID, DEMO_MOVIE_ID))
print("Predicted Rating:", prediction)


SINGLE RATING PREDICTION
User : 25
Movie: Toy Story (1995)
Predicted Rating: 5.0


STEP 5 - Generate TOP N Recommendations for a User

In [35]:
def recommend_basic_item_cf(user_id, top_n=TOP_N, top_k=TOP_K):

    if user_id not in train_matrix.index: return pd.DataFrame()

    predictions = []

    for movie_id in train_matrix.columns:

        if pd.notna(train_matrix.loc[user_id, movie_id]): continue

        prediction = predict_basic_item_cf(user_id, movie_id, top_k=top_k)

        if pd.notna(prediction): predictions.append((movie_id, prediction))

    result = pd.DataFrame(predictions, columns=["movieId", "Predicted Rating"])

    if result.empty: return result

    result["Movie"] = result["movieId"].map(movie_titles)

    return (result.sort_values("Predicted Rating", ascending=False).head(top_n)[["movieId", "Movie", "Predicted Rating"]])

In [36]:
print("\n" + "=" * 70)
print("TOP-N ITEM-BASED RECOMMENDATIONS")
print("=" * 70)

display(recommend_basic_item_cf(DEMO_USER_ID))


TOP-N ITEM-BASED RECOMMENDATIONS


,movieId,Movie,Predicted Rating
0,1,Toy Story (1995),5.0
1,6,Heat (1995),5.0
4,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),5.0
9,110,Braveheart (1995),5.0
5,36,Dead Man Walking (1995),5.0
6,47,Seven (a.k.a. Se7en) (1995),5.0
7,50,"Usual Suspects, The (1995)",5.0
42,1193,One Flew Over the Cuckoo's Nest (1975),5.0
40,1073,Willy Wonka & the Chocolate Factory (1971),5.0
45,1200,Aliens (1986),5.0


STEP 6 - Evaluate Rating Prediction

In [ ]:
actual = []
predicted = []

for row in test_data.itertuples():

    prediction = predict_basic_item_cf(row.userId, row.movieId)

    if pd.notna(prediction):
        actual.append(row.rating)
        predicted.append(prediction)

coverage = len(predicted) / len(test_data)

In [38]:
print("\n" + "=" * 70)
print("BASIC ITEM-CF EVALUATION")
print("=" * 70)

print("Total test ratings :", len(test_data))
print("Predictions made   :", len(predicted))
print(f"Prediction coverage: {coverage * 100:.2f}%")


BASIC ITEM-CF EVALUATION
Total test ratings : 20168
Predictions made   : 14371
Prediction coverage: 71.26%


In [ ]:
if predicted:

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load data
ratings = pd.read_csv("mlsmall/ratings.csv")
movies = pd.read_csv("mlsmall/movies.csv")

# 80-20 split
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# User-Movie matrix
R = train.pivot(index="userId",columns="movieId",values="rating")

# Mean-center ratings
mean = R.mean(axis=1)
Rc = R.sub(mean, axis=0).fillna(0)

# --------------------------------------------------
# Item-Item similarity
# Transpose because we compare movies, not users
# --------------------------------------------------

sim = cosine_similarity(Rc.T)
sim = pd.DataFrame(sim, index=R.columns, columns=R.columns)

# --------------------------------------------------
# Prediction function
# --------------------------------------------------

def predict(uid, movie):

    if uid not in R.index or movie not in R.columns: return mean.mean()

    # Movies rated by the user
    rated = R.loc[uid].dropna()
    # 10 most similar movies
    neighbors = sim.loc[movie, rated.index].nlargest(10)

    num = den = 0

    for m, s in neighbors.items():
        num += s * (R.loc[uid, m] - mean[uid])
        den += abs(s)
    return mean[uid] + num / den if den else mean[uid]


# --------------------------------------------------
# Predict train and test ratings
# --------------------------------------------------

train_pred = [predict(u, m) for u, m in zip(train.userId, train.movieId)]
test_pred = [predict(u, m) for u, m in zip(test.userId, test.movieId)]


# --------------------------------------------------
# Evaluation
# --------------------------------------------------

print("Train MAE :", mean_absolute_error(train.rating, train_pred))
print("Train RMSE:", np.sqrt(mean_squared_error(train.rating, train_pred)))
print("Test MAE  :", mean_absolute_error(test.rating, test_pred))
print("Test RMSE :", np.sqrt(mean_squared_error(test.rating, test_pred)))


# --------------------------------------------------
# Top-10 recommendations
# --------------------------------------------------

uid = 25
top_k = 10

pred = {}

for movie in R.columns[R.loc[uid].isna()]: pred[movie] = predict(uid, movie)

top10 = sorted( pred.items(), key=lambda x: x[1], reverse=True)[:top_k]

print("\nTop 10 Recommendations:")

for movie, rating in top10:
    title = movies.loc[movies.movieId == movie, "title"].iloc[0]
    print(title, ":", round(rating, 2))

FileNotFoundError: [Errno 2] No such file or directory: 'mlsmall/ratings.csv'